# Example: predict the science-arm sky spectrum from minimal inputs

This notebook shows how to use a trained ensemble to predict the sky
spectrum at the science pointing from the smallest possible per-row input:

1. The two sky-arm decomposition coefficient vectors (from the QP
   decomposition, one per sky arm), and
2. Pointing metadata that cannot be derived from anything else:
    - `obstime_mjd` (UT MJD),
    - `sci_ra`, `sci_dec` (science pointing, degrees),
    - `sky_e_ra`, `sky_e_dec` (SkyE sky-arm pointing, degrees),
    - `sky_w_ra`, `sky_w_dec` (SkyW sky-arm pointing, degrees).

The near-vs-far arm assignment (what the model actually consumes) is
computed internally from angular separation to the science pointing.

Everything else (astrometry, moon/sun ephemerides, van Rhijn slant-path
factors, ecliptic geometry, physics-prior moon-scatter proxies, the
space-weather indices, and the Phase-A' interaction features — 42 context
features per arm in total) is computed inside
`mlp_predictor.inference.predict_sky_from_minimal_inputs`.

**Prerequisite**: run the training notebook
`notebook_sky_interpolation_triplet_dual_encoder_group_mlp_split_zodi_module.ipynb`
to fit the ensemble.  The save cell at the end of that notebook writes the
trained ensemble to a `.pt` archive that this notebook then loads.

## 1. Load the trained ensemble

In [ ]:
import os
os.environ.setdefault("LVMCORE_DIR", "/Users/droryn/prog/lvm/lvmcore")

import numpy as np
from mlp_predictor import config, data, serialization, inference

# The deployed corpus and ensemble as of 2026-09-25: DRP 1.3.2 medians decomposed
# with `palacecorr-aijc-vnf-split-zodi-lsf-spline2d` (telluric fit, ridge-corrected
# PALACE OH line strengths) and the lvm-ecl-2026-09 zodiacal-light correction,
# every decompose_parallel setting at its default.  `config.PipelineConfig()`
# still defaults to an older corpus, so set it explicitly rather than inheriting it.
DECOMP_VARIANT = "telluric-palacecorr"
cfg = config.PipelineConfig()
cfg.data.decomp_data_root = "gaia-stars-mask-telluric-chi2-palacecorr"
cfg.data.decomp_stem = "lvmsframe_median_stack_1.3.2_gaia1over100"
cfg.data.decomp_suffix = data.DECOMP_VARIANTS[DECOMP_VARIANT]["suffix"]
ENSEMBLE_PATH = f"{cfg.data.decomp_data_root}/mlp_ensemble_split_zodi_cont-1.3.2.pt"
print(f"loading ensemble from: {ENSEMBLE_PATH}")

ensemble = serialization.load_ensemble(ENSEMBLE_PATH)
print(f"loaded {len(ensemble['members'])}-seed ensemble; "
      f"deployed config knobs:")
for k in ('moon_zodi_mode', 'alpha_ctx_groups', 'flux_mse_groups',
         'zodi_ctx_restriction', 'continuum_ctx_restriction'):
    _v = ensemble['config'].get(k)
    print(f"  {k}: {_v}")
print(f"zodi correction the targets were anchored with: {ensemble.get('zodi_correction', 'none')}")
print(f"predicted coefficient dim: {len(ensemble['coef_names'])}")
print(f"context features per arm: {len(ensemble['ctx_names'])}")

## 2. Required per-row inputs

The prediction function needs the following minimal set per row.  Angles
are ICRS degrees; MJD is UT.

| input | shape | meaning |
|---|---|---|
| `obstime_mjd`  | `(n_rows,)`   | MJD (UT) of the exposure — sole source of all time features and moon/sun ephemerides |
| `sci_ra`, `sci_dec`         | `(n_rows,)` each | science-fiber pointing (deg) |
| `sky_e_ra`, `sky_e_dec`     | `(n_rows,)` each | SkyE sky-arm pointing (deg) |
| `sky_w_ra`, `sky_w_dec`     | `(n_rows,)` each | SkyW sky-arm pointing (deg) |
| `coef_e` | `(n_rows, n_coef=388)` | decomposition coefficients from the SkyE spectrum |
| `coef_w` | `(n_rows, n_coef=388)` | decomposition coefficients from the SkyW spectrum |

The near-vs-far dispatch (which arm is closer to the science pointing per row)
is computed internally.  The returned dict includes a
`near_is_east` boolean per row so downstream code can see which arm was
picked.

**Optional overrides**:

| input | default | meaning |
|---|---|---|
| `moon_phase_deg` | computed via astropy | moon phase (0=new, 180=full).  Providing this manually avoids one astropy roundtrip. |

| `wave` | `(n_pix,)` | the wavelength grid of the LSFs below (Å, air) |
| `lsf_e`, `lsf_w`, `lsf_sci` | `(n_pix,)` or `(n_rows, n_pix)` | LSF FWHM in Å for the SkyE, SkyW and science fibres |
| `date_obs` | ISO-8601 string, or one per row | exposure timestamp, e.g. `2024-03-11T02:14:07.5` |
| `expnum` | int, optional | exposure number; only used for error messages from the moon/zodi model |

The last four are needed because three of the 42 context features —
`moon_model_log_ratio`, `zodi_po_log10`, `moon_frac_po` — come from the
moon/zodi physical model rather than from geometry alone, and it needs the
resolution and the timestamp. In the training pipeline they are read from a
per-corpus cache; here they are evaluated directly for the pointings you pass.
Omitting them raises an error that names the features it could not build.

Note `obstime_mjd` must be the **exposure** timestamp, not the observing-night
integer MJD. The corpus META carries both (`obstime` and `mjd`); the night
number puts every exposure at midnight UT, which moves the moon and can put the
target below the horizon.

**Not required** (derived internally): everything else in the 42-feature
context — `alt`, `az_{sin,cos}`, `airmass`, `moon_alt`, `moon_sep`,
`moon_phase_{sin,cos}`, `moon_az_{sin,cos}`, `sun_{sep,alt}`,
`sun_az_{sin,cos}`, `sci_sep`, `vanrhijn_{87km,95km,285km}`,
`obstime_{day,lunation,year}_{sin,cos}`, `f107`, `f107_81d`, `kp`,
`ecl_beta_deg`, `ecl_lon_{sin,cos}`, `zodi_log10_v`, `moon_fli`,
`moon_up_smooth`, `moon_airmass_up`, `moon_signal_proxy`,
`moon_fli_x_phase_cos`, `moon_sig_x_lon_{cos,sin}`.

See intro §3.4.5 in the training notebook for how each of those enters the
model.

## 3. Build inputs for model and predict sky spectrum at science IFU

Example code builds simple medians of the SkyE, SkyW and science fibres and
decomposes the two sky medians.  The decomposition is the one the training corpus
was built with, run through `decompose_parallel.decompose_in_process` with every
setting at its `decompose_parallel` default (fit model
`palacecorr-aijc-vnf-split-zodi-lsf-spline2d`).  The decomposition coefficients
are what is fed to the predictor later.

In [ ]:
# --- Load and prep inputs from a real LVM observation ---
# Open a reduced lvmCFrame, use SLITMAP to isolate the SkyE and SkyW
# fibres, median-stack each into a per-arm sky spectrum, decompose both
# EXACTLY as the training corpus was decomposed (see below), and package the
# results as the (1, 388) coefficient batch that `predict_sky_from_minimal_inputs`
# consumes.  The near-vs-far assignment is done inside that function.
# Also median-stacks the CFrame LSF across the SCI fibres into
# `sci_lsf_fwhm_ang` (Angstrom FWHM per pixel) for use in §5.

import sys
from pathlib import Path

from astropy.io import fits
from astropy.table import Table
from astropy.time import Time

from mlp_predictor import wavelengths as _wave_mod
from mlp_predictor.data import _infer_base_dir_for_reconstruction

CFRAME_PATH = (
    "/Users/droryn/work/LVM/data/sas/sdsswork/lvm/spectro/redux/1.2.2dev/"
    "1046XX/1046750/60356/lvmCFrame-00012630.fits"
)
FACTOR = 1e14  # CFrame flux is erg/s/cm^2/A; the QP fits O(1) counts.

with fits.open(CFRAME_PATH) as hdul:
    hdr  = hdul['PRIMARY'].header
    wave = np.asarray(hdul['WAVE'].data, dtype=np.float64)
    flux = np.asarray(hdul['FLUX'].data, dtype=np.float64)
    ivar = np.asarray(hdul['IVAR'].data, dtype=np.float64)
    lsf  = np.asarray(hdul['LSF'].data,  dtype=np.float64)   # FWHM in Angstroms
    slit = hdul['SLITMAP'].data

# Select good SkyE and SkyW fibres (fibstatus == 0 = usable).
_tel = np.asarray(slit['telescope']).astype(str)
_fst = np.asarray(slit['fibstatus']).astype(int)
skye_fibers = (_tel == 'SkyE') & (_fst == 0)
skyw_fibers = (_tel == 'SkyW') & (_fst == 0)
sci_fibers  = (_tel == 'Sci')  & (_fst == 0)
sci_lsf_fwhm_ang = np.nanmedian(lsf[sci_fibers], axis=0)   # (n_pix,) FWHM in Å, med across SCI
# Per-arm LSF too.  The moon/zodi model needs the resolution of the fibre it is
# predicting for, and `predict_sky_from_minimal_inputs` swaps the two sky arms
# per row when it decides which is near -- so each LSF has to travel with its
# own arm, exactly like coef_e / coef_w.
skye_lsf_fwhm_ang = np.nanmedian(lsf[skye_fibers], axis=0)
skyw_lsf_fwhm_ang = np.nanmedian(lsf[skyw_fibers], axis=0)

def _median_stack(fiber_mask):
    """Median-combine flux, propagate ivar under Gaussian-median statistics."""
    _f  = flux[fiber_mask]
    _iv = ivar[fiber_mask]
    _valid = np.isfinite(_f) & np.isfinite(_iv) & (_iv > 0.0)
    _f_masked = np.where(_valid, _f, np.nan)
    _sigma2   = np.where(_valid, 1.0 / _iv, np.nan)
    med_flux    = np.nanmedian(_f_masked, axis=0)
    n_valid     = np.sum(_valid, axis=0)
    mean_sigma2 = np.nanmean(_sigma2, axis=0)
    # For n Gaussian samples, Var(median) ~ (pi/2n) * sigma^2 (large-n asymptote).
    med_ivar = np.where(
        (n_valid > 0) & np.isfinite(mean_sigma2) & (mean_sigma2 > 0.0),
        (2.0 * n_valid) / (np.pi * np.maximum(mean_sigma2, 1e-300)),
        0.0,
    )
    return med_flux, med_ivar

flux_east, ivar_east = _median_stack(skye_fibers)
flux_west, ivar_west = _median_stack(skyw_fibers)
# The science median as well: the decomposition centres its science emission-line
# mask on the Halpha velocity measured from it, for every arm.
flux_sci_med, _ = _median_stack(sci_fibers)
print(f"stacked SkyE across {int(skye_fibers.sum())} good fibres; "
      f"SkyW across {int(skyw_fibers.sum())}; Sci across {int(sci_fibers.sum())}.")

# --- Decompose the two stacked sky spectra EXACTLY as the corpus was ---------
# The coefficients handed to the predictor must come from the SAME
# decomposition the training corpus was built with, or they sit on a different
# footing from everything the network was trained on.  So this does not
# construct a decomposer here: it builds the exposure as a one-row stack in the
# corpus format, in memory (an HDUList, no file), and hands it to
# `decompose_parallel.decompose_in_process`, which
# runs the same worker code a default cluster run does -- fit model
# (palacecorr), knots, smoothing, every identifiability constraint, the zodi
# correction, photon pixel weights, the science-line mask centred on the
# measured Halpha velocity, this exposure's telluric transmission (PWV and
# airmasses), the geometry amplitude priors and the reversal retry, all at the
# decompose_parallel defaults.  The spectra are stored in physical units; the
# worker applies FACTOR itself.  The META row is built by lvm_medians'
# `median_meta` from the frame header, the function that writes the corpus META.
import decompose_parallel as _dp
sys.path.insert(0, str(Path("medians_computation/src").resolve()))
from lvm_medians.metadata import median_meta
from lvm_medians.stack import _coords, _separation

def build_one_row_stack(header, spectra, lsfs, fiber_counts):
    """One exposure as an in-memory one-row stack that decompose_parallel reads.

    `spectra` / `lsfs`: dicts keyed "Sci", "SkyE", "SkyW" with (n_pix,) flux in
    erg/s/cm^2/A and LSF FWHM in A.  The near/far assignment follows lvm_medians
    (the sky telescope closer to the science pointing is "near").
    Returns ``(hdulist, near, far)``.
    """
    positions = {n: _coords(header, n) for n in ("Sci", "SkyE", "SkyW")}
    separations = {n: _separation(positions["Sci"], positions[n]) for n in ("SkyE", "SkyW")}
    near = min(separations, key=lambda k: separations[k] if np.isfinite(separations[k]) else np.inf)
    far = "SkyW" if near == "SkyE" else "SkyE"
    counts = {}
    for n, c in fiber_counts.items():
        counts[f"{n}_good"] = counts[f"{n}_used"] = int(c)
    meta = median_meta(header=header, path=Path(CFRAME_PATH), input_index=0,
                       expnum=int(header["EXPOSURE"]), positions=positions,
                       separations=separations, near=near, far=far, counts=counts,
                       sci_percentile=np.nan, sky_percentile=np.nan,
                       gaia_sigma=None, gaia_ratio_threshold=None)
    img = lambda a, name: fits.ImageHDU(np.asarray(a, dtype=np.float32)[None, :], name=name)
    stack = fits.HDUList([
        fits.PrimaryHDU(),
        fits.ImageHDU(np.asarray(wave, dtype=np.float64), name="WAVE"),
        img(spectra["Sci"], "FLUX_SCI"),
        img(spectra[near], "FLUX_SKY_NEAR"),
        img(spectra[far], "FLUX_SKY_FAR"),
        img(lsfs["Sci"], "LSF_SCI"),
        img(lsfs[near], "LSF_SKY_NEAR"),
        img(lsfs[far], "LSF_SKY_FAR"),
        fits.BinTableHDU(Table(rows=[meta]), name="META"),
    ])
    return stack, near, far

_expnum = int(hdr['EXPOSURE'])
stack, near_label, far_label = build_one_row_stack(
    hdr,
    spectra={"Sci": flux_sci_med, "SkyE": flux_east, "SkyW": flux_west},
    lsfs={"Sci": sci_lsf_fwhm_ang, "SkyE": skye_lsf_fwhm_ang, "SkyW": skyw_lsf_fwhm_ang},
    fiber_counts={"Sci": sci_fibers.sum(), "SkyE": skye_fibers.sum(), "SkyW": skyw_fibers.sum()})
print(f"one-row in-memory stack: near = {near_label}, far = {far_label}")

_fits = _dp.decompose_in_process(stack, rows=(0,), kinds=("sky1", "sky2"))
fit_near, flags_near = _fits[("sky1", 0)]
fit_far, flags_far = _fits[("sky2", 0)]
fit_east, fit_west = (fit_near, fit_far) if near_label == "SkyE" else (fit_far, fit_near)
for _label, _fit, _flags in ((near_label, fit_near, flags_near), (far_label, fit_far, flags_far)):
    print(f"{_label} decomp: status={_fit.fit_status}, chi2_red={_fit.reduced_chi2:.3f} "
          f"(photon-weighted), R^2={_fit.r2:.4f}, reliability={_flags.get('reliability')}")
print(f"decomposition: fit model {_dp._WORKER_FIT_MODEL}, zodi correction "
      f"{_dp.SPLIT_ZODI_ZODI_CORRECTION}")

N_MOON_KNOTS, SPLIT_ZODI, N_ZODI_KNOTS = _wave_mod.infer_spline_knots(ensemble['coef_names'])
assert fit_east.coef.size == len(ensemble['coef_names']), "decomposition and ensemble disagree on the basis"

# --- Package inputs for `predict_sky_from_minimal_inputs` (batch of 1) ---
obstime_mjd = np.array([Time(str(hdr['OBSTIME']), format='isot', scale='utc').mjd])
sci_ra      = np.array([float(hdr['SCIRA'])])
sci_dec     = np.array([float(hdr['SCIDEC'])])
sky_e_ra    = np.array([float(hdr['SKYERA'])])
sky_e_dec   = np.array([float(hdr['SKYEDEC'])])
sky_w_ra    = np.array([float(hdr['SKYWRA'])])
sky_w_dec   = np.array([float(hdr['SKYWDEC'])])
coef_e = fit_east.coef.astype(np.float32)[None, :]
coef_w = fit_west.coef.astype(np.float32)[None, :]

print()
print(f"sci pointing: RA={sci_ra[0]:.4f}, Dec={sci_dec[0]:.4f}, MJD={obstime_mjd[0]:.4f}")
print(f"SkyE arm:     RA={sky_e_ra[0]:.4f}, Dec={sky_e_dec[0]:.4f}")
print(f"SkyW arm:     RA={sky_w_ra[0]:.4f}, Dec={sky_w_dec[0]:.4f}")
print(f"batch shapes: coef_e={coef_e.shape}, coef_w={coef_w.shape}")

In [ ]:
# Run the ensemble prediction.  Auto-computes all 42 ctx features, works out
# which sky arm is near vs far from the SkyE/SkyW pointings, and averages across
# the 10 seeds.
#
# `wave`, the three LSFs and `date_obs` are REQUIRED by this ensemble: three of
# its 42 context features (`moon_model_log_ratio`, `zodi_po_log10`,
# `moon_frac_po`) come from the moon/zodi physical model, which needs the
# resolution and the exposure timestamp.  In the training pipeline they are read
# from a per-corpus cache; for arbitrary pointings they are evaluated directly.
# Omit them and the call raises, naming the features it could not build.
#
# `obstime_mjd` must be the EXPOSURE timestamp, not the observing-night integer
# MJD -- the corpus META carries both (`obstime` and `mjd`) and the night number
# puts every exposure at midnight UT, which silently moves the moon and can even
# put the target below the horizon.
result = inference.predict_sky_from_minimal_inputs(
    ensemble,
    obstime_mjd=obstime_mjd,
    sci_ra=sci_ra,     sci_dec=sci_dec,
    sky_e_ra=sky_e_ra, sky_e_dec=sky_e_dec,
    sky_w_ra=sky_w_ra, sky_w_dec=sky_w_dec,
    coef_e=coef_e,     coef_w=coef_w,
    wave=wave,
    lsf_e=skye_lsf_fwhm_ang, lsf_w=skyw_lsf_fwhm_ang, lsf_sci=sci_lsf_fwhm_ang,
    date_obs=str(hdr['OBSTIME']).strip(),
    expnum=int(hdr.get('EXPOSURE', hdr.get('EXPNUM', -1))),
)
print("result keys:", list(result.keys()))
_near_arm = np.where(result['near_is_east'], 'SkyE', 'SkyW')
print(f"per-row near arm: {_near_arm}")
for k in ('coef', 'coef_std', 'confidence', 'reliability'):
    v = result[k]
    print(f"  {k}: shape={v.shape} dtype={v.dtype}")


## 4. Return-value structure

`predict_sky_from_minimal_inputs` returns a dict with the following keys:

| key | shape | meaning |
|---|---|---|
| `coef`         | `(n_rows, 388)` `float32` | **The prediction.** Ensemble-mean science-arm decomposition coefficients, already multiplied by the Jensen per-coefficient bias-lift (§3.6.6 of the training doc).  Reconstruct the physical flux spectrum by matrix-multiplying with the corresponding basis matrix from `SkyDecompLSFSurfaceIterative` (see §5 below). |
| `coef_std`     | `(n_rows, 388)` `float32` | **Per-coefficient epistemic uncertainty.**  Standard deviation across the 10 ensemble seeds — a first-order proxy for the model's confidence in each individual coefficient. |
| `confidence`   | `(n_rows,)` `float32` in (0, 1] | **Row-level confidence score.**  Defined as $1 / (1 + \mathrm{median}_k \ (\sigma_k / |c_k|))$ across the coefficient dimension.  1.0 = perfect seed agreement; ~0.5 = median relative std of ~100%; ~0.1 = the seeds disagree wildly.  See §4.1 below for how to read it. |
| `reliability`  | `(n_rows,)` `int32` | **Per-row reliability bitmask**, same vocabulary as the decomposition products (`sky_decomp/reliability.py`).  Errors live in the low 16 bits, warnings from bit 16 up; `-1` means the test could not be evaluated at all.  See §4.2. |
| `coef_names`   | list of 388 strings | Coefficient identifiers matching the columns of `coef` and `coef_std`, e.g. `OH_004`, `Moon_bs02`, `Zodi_bs01`. |
| `triplet`      | dict                | The 42-dim per-arm context that was fed to the model, plus the raw RA/Dec/MJD, useful for debugging or feeding a second predictor. |

Pass `return_per_seed=True` to additionally get a `per_seed` key with the
full `(10, n_rows, 388)` per-seed prediction cube.

### 4.1 Reading the confidence score

The scalar `confidence` in each row is a monotone rescaling of the median
*relative* seed spread across the 388 coefficient dimensions:

$$\mathrm{confidence}(r) \;=\; \frac{1}{1 + \mathrm{median}_k \bigl( \sigma_k(r) / |\bar{c}_k(r)| \bigr)}.$$

Practical reading:

* **`confidence \ge 0.9`** — the 10 seeds agree to better than ~10 % relative
  on the median coefficient; the model considers the row well-covered by
  the training distribution.  Typical for moon-down / low-airmass rows.
* **`0.6 \le confidence < 0.9`** — moderate seed spread.  The row sits near
  the edge of the training distribution or in a regime where the physics
  itself is variable (bright-moon-close, close_zodi, gravity-wave-rich
  nights).  The prediction is still usable but the per-pixel error bar is
  wider.
* **`confidence < 0.6`** — the seeds actively disagree.  The row is in a
  regime the model has not seen enough of during training, or the input
  coefficients are outside the coverage of the training corpus.  Treat the
  prediction as a rough guess and consider flagging it downstream.

For downstream code that needs a full covariance rather than a scalar
summary, use `coef_std` directly — it gives the per-coefficient 1-sigma
spread across the ensemble, on the same physical scale as `coef`.

In [ ]:
# Show the confidence per row and highlight which coefficients are
# most/least uncertain.
for r in range(result['coef'].shape[0]):
    c   = result['coef'][r]
    std = result['coef_std'][r]
    conf = float(result['confidence'][r])
    # Relative std, guarding against tiny |c|.
    rel = std / np.maximum(np.abs(c), 1e-12)
    order = np.argsort(rel)
    worst = order[-3:][::-1]
    best  = order[:3]
    print(f"row {r}: confidence = {conf:.3f}")
    print(f"   most uncertain coefs: ", end='')
    print(', '.join(f"{result['coef_names'][k]}={c[k]:+.3g}±{std[k]:.2g}" for k in worst))
    print(f"   most certain coefs:   ", end='')
    print(', '.join(f"{result['coef_names'][k]}={c[k]:+.3g}±{std[k]:.2g}" for k in best))

### 4.2 Reading the reliability bitmask

`reliability` reports the pathologies that the corpus build handles by
DROPPING a row.  Production sky subtraction cannot drop a row -- it still owes
a spectrum -- so the flags are carried instead.  The vocabulary is shared with
the decomposition products (`sky_decomp/reliability.py`), and it is split by
severity:

| severity | bits | how to read it |
|---|---|---|
| **error** (low 16 bits) | `fit_failed`, `reversed`, `diffuse_collapsed`, `sci_colour_excess` | The coefficients do not describe what their names say.  Do not score, train on, or trust the per-family split of such a row. |
| **warning** (bit 16 and up) | `reversal_untestable`, `reversal_retried`, `reversal_recovered`, `zodi_anchor_pinned`, `moon_share_pinned`, `diffuse_oh_cap_binding`, `diffuse_ratio_pinned`, `shape_bound_active` | A CONSTRAINT shaped the fit, so the value is partly prior rather than data.  Informational: not a reason to reject the row. |

Two things to know about the PREDICTION side specifically:

* **Only `diffuse_collapsed` is computable from coefficients alone.**  It is
  referenced to the ensemble's own `coef_upper_bound`, and selects the same
  rows as the decomposition-side corpus gate.
* **`reversed` needs the reconstructed continua**, because it is a statement
  about the moon and zodi COLOURS.  §5 reconstructs the components anyway to
  build the sky spectrum, so the bit is added there with
  `inference.reliability_from_components`.
* **The constraint warnings never appear here.**  There is no QP at prediction
  time; those bits describe how the row's TRAINING TARGETS were shaped, which
  is a property of the corpus, not of this prediction.

`-1` is distinct from `0`: it means the test could not be evaluated (no usable
reference), and must not be read as a clean row.

In [ ]:
# Read the prediction's reliability bitmask and print the clear name of every
# bit that is set.  `rel.RELIABILITY_BITS` is the single source of truth for the
# bit -> name mapping, so this stays correct if bits are appended later.
from sky_decomp import reliability as rel

def report_reliability(bits, label=''):
    """Print a row's reliability bits by name, split into errors and warnings."""
    bits = int(bits)
    if bits < 0:
        print(f"{label}reliability = {bits} -> NOT EVALUATED "
              f"(no usable reference; this is NOT the same as 'clean')")
        return
    errors   = [name for value, name in rel.RELIABILITY_BITS
                if (bits & value) and (value & rel.RELIABILITY_ERROR_MASK)]
    warnings = [name for value, name in rel.RELIABILITY_BITS
                if (bits & value) and (value & rel.RELIABILITY_WARNING_MASK)]
    verdict = ('ERROR' if rel.has_error(bits)
               else 'warning' if rel.has_warning(bits) else 'clean')
    print(f"{label}reliability = 0x{bits:06x} ({bits})  ->  {verdict}")
    print(f"{label}   errors  : {', '.join(errors) if errors else '(none)'}")
    print(f"{label}   warnings: {', '.join(warnings) if warnings else '(none)'}")
    # rel.describe() gives the same thing as one compact string, for logs.
    print(f"{label}   compact : {rel.describe(bits)}")

for r in range(result['coef'].shape[0]):
    report_reliability(result['reliability'][r], label=f"row {r}: ")

# The full vocabulary, so a set bit can always be looked up from the output.
print("\nknown bits:")
for value, name in rel.RELIABILITY_BITS:
    kind = 'error  ' if value & rel.RELIABILITY_ERROR_MASK else 'warning'
    print(f"  0x{value:06x}  {kind}  {name}")

## 5. Reconstruct the predicted sky spectrum

The prediction is a coefficient vector; the flux spectrum is
$f(\lambda) = A \hat{\mathbf{c}}$ where $A$ is the design matrix from
the decomposition side (§1.3 of the training doc).  We reconstruct that
here on the deployed wavelength grid with the corpus's own basis -- the
palacecorr telluric decomposer, ridge-corrected OH line strengths and this
exposure's transmission along the science line of sight -- built by
`data.make_reconstruction_decomposer` exactly as the training diagnostics build it.

### Note that an open question here is how we get the LSF vector for the data
for now, we just read the stored LSF, but we know this is not good enough in drp-1.3.1

In [ ]:
# Reconstruct the predicted science-arm sky on the SAME basis the corpus was
# fitted with: the palacecorr telluric decomposer -- ridge-corrected OH line
# strengths, this exposure's DRP transmission along the science line of sight --
# built by `data.make_reconstruction_decomposer` exactly as the training
# diagnostics build it.  The OH line file travels in the telluric bundle, chosen
# from the corpus suffix, so it cannot silently fall back to the canonical one.
# The science arm itself was not decomposed, so its LSF is the CFrame's median
# SCI LSF installed as a nominal (Gaussian-width) surface.
from sky_decomp.moon_zodi_model import LSF_FWHM_TO_SIGMA

_meta_tab = Table(stack["META"].data)
_tel_sci = data.telluric_row_kwargs(
    _meta_tab, 0, "sci", wave, sci_lsf_fwhm_ang,
    palace_oh_suffix=data.palace_oh_suffix_for(cfg.data.decomp_suffix))
_recon = data.make_reconstruction_decomposer(
    wave, n_spline_knots=N_MOON_KNOTS, base_dir=_infer_base_dir_for_reconstruction(),
    split_zodi=SPLIT_ZODI, n_zodi_spline_knots=N_ZODI_KNOTS, telluric=_tel_sci,
    lsf_sigma=sci_lsf_fwhm_ang / LSF_FWHM_TO_SIGMA)
_recon._set_lsf_state(_recon._nominal_state("example_sci_nominal_lsf"))
mats = _recon._assemble_refined_matrices()
# The O2 b band's template is a per-row pre-fit shape (VECTOR_O2).  The science
# arm has none of its own here, so use the near arm's fitted shape: same
# atmosphere, same instant.
mats["o2"] = np.asarray(fit_near.vector_o2, dtype=np.float64).ravel()[None, :]
print(f"reconstruction basis: {type(_recon).__name__}, OH file suffix "
      f"{data.palace_oh_suffix_for(cfg.data.decomp_suffix)!r}, "
      f"SCI LSF FWHM median {np.median(sci_lsf_fwhm_ang):.3f} Å")

def _sum_components(comps):
    # Sum per-family fluxes; 'zodi' is only present when split_zodi=True.
    total = np.zeros_like(np.asarray(comps['oh'], dtype=np.float64))
    for key in ('oh', 'moon', 'zodi', 'diffuse', 'atom', 'orc', 'o2'):
        arr = comps.get(key)
        if arr is not None:
            total = total + np.asarray(arr, dtype=np.float64)
    return total

# Row 0 reconstruction (mean coefficients + ensemble std envelope).
coef0    = result['coef'][0].astype(np.float64)
coef0_lo = coef0 - result['coef_std'][0].astype(np.float64)
coef0_hi = coef0 + result['coef_std'][0].astype(np.float64)
comps0    = _recon._components_from_coef(coef0, mats)
flux_mean = _sum_components(comps0)
flux_lo   = _sum_components(_recon._components_from_coef(np.maximum(coef0_lo, 0), mats))
flux_hi   = _sum_components(_recon._components_from_coef(np.maximum(coef0_hi, 0), mats))

print(f"reconstructed spectrum: n_lambda={flux_mean.size}, "
      f"mean flux range=[{flux_mean.min():.3g}, {flux_mean.max():.3g}]")

# Now that the components exist, add the reliability bits that need them: a
# moon/zodi role REVERSAL is a statement about the two fitted CONTINUA, so it
# cannot be seen in the coefficient vector alone (§4.2).  OR it into the
# coefficient-only bitmask the prediction already returned.
_rev_bits, _rev_info = inference.reliability_from_components(comps0, _recon.wave)
bits_row0 = int(result['reliability'][0]) | int(_rev_bits)
print(f"\nmoon slope {_rev_info['moon_slope']:+.3f}, "
      f"zodi slope {_rev_info['zodi_slope']:+.3f}, "
      f"separation {_rev_info['separation']:+.3f} "
      f"(negative = reversed), moon share {_rev_info['moon_frac']:.4f}")
report_reliability(bits_row0, label='row 0 (with reconstruction): ')

In [ ]:
# Plot the reconstructed sky spectrum with a shaded ensemble-uncertainty band.
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=wave, y=flux_hi, mode='lines',
                          line=dict(color='rgba(30, 30, 200, 0.0)'),
                          showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter(x=wave, y=flux_lo, mode='lines',
                          line=dict(color='rgba(30, 30, 200, 0.0)'),
                          fill='tonexty', fillcolor='rgba(30, 30, 200, 0.15)',
                          name='±1σ ensemble spread'))
fig.add_trace(go.Scatter(x=wave, y=flux_mean, mode='lines',
                          line=dict(color='rgb(30, 30, 200)', width=1.0),
                          name='predicted sky flux (row 0)'))
fig.update_layout(xaxis_title='wavelength [Å]',
                   yaxis_title='flux (native decomp units)',
                   height=400, margin=dict(l=60, r=20, t=40, b=50),
                   title=f"row 0: confidence={result['confidence'][0]:.3f}")
fig.show()

## 6. Where to look next

- **`mlp_predictor.inference.build_triplet_from_pointings`** — the ctx
  builder called under the hood.  Returns the 42-dim per-arm context if
  you want to feed it to `predict_sci_coefficients_default` directly.
- **`mlp_predictor.trainer.predict_sci_coefficients_default(artifact, ...)`** —
  the low-level predictor.  Bypasses the ensemble averaging so you can
  probe per-seed predictions if you want a specific seed's output.
- **Training notebook §3.4.5** — a table of which of the 42 context
  features enter which specialised branch of the model.
- **Training notebook §3.4.3** — the derivation of the Phase-F additive
  moon-zodi coupling that the ensemble uses; explains why the additive
  residual is zero at $t=0$ and where the moon/zodi predictions get their
  cross-group information.
- **`sky_decomp.lsf_surface_iterative.SkyDecompLSFSurfaceIterative`** —
  the same class used above to reconstruct the flux spectrum.  Use its
  `_components_from_coef` on your predicted `coef` to get the per-family
  breakdown (moon, zodi, OH lines, atomic, diffuse continuum, O₂) if you
  need to subtract only a subset.

## 7. Decompose a raw sky spectrum

The prediction pipeline (§§1–5) consumes `coef_near` / `coef_far` vectors
that are produced by the physical decomposition described in Chapter 1 of the
training notebook.  If you have a **raw sky spectrum**, this section shows how
to run the same decomposition on it.  The result is the 388-dimensional
coefficient vector you would then feed to `predict_sky_from_minimal_inputs`
for a different pointing (or reconstruct back to flux as in §5).

The deployed decomposition needs the exposure's metadata, not just the
spectrum: the telluric transmission uses its PWV and airmasses, the amplitude
priors its pointing and timestamp, and the science-line mask is centred on the
Halpha velocity measured from the same exposure's science spectrum.  The cell
therefore puts the spectrum and header into an in-memory one-row stack
(`build_one_row_stack`, §3) and calls `decompose_parallel.decompose_in_process`,
the same code and defaults as the cluster run.

**Assumed inputs** (bind these before running the cell below; the defaults
decompose the SkyE median from §3):

| variable | shape | meaning |
|---|---|---|
| `raw_flux` | `(n_pix,)` `float64` | sky flux in erg/s/cm²/Å on `wave` (not scaled by `FACTOR`) |
| `raw_lsf` | `(n_pix,)` `float64` | its LSF FWHM in Å |
| `raw_header` | FITS header | the exposure's primary header (pointings, `OBSTIME`, airmasses, `PWV_MED`) |

The fit runs the O2 pre-fit, the joint seed QP and five refinement cycles
(continuum → LSF → lines) with the telluric design rebuilt for the row --
a few seconds per spectrum on a laptop.

In [ ]:
# --- Decompose a raw sky spectrum ---
#
# The deployed decomposition is not a free-standing fit of a spectrum: its
# telluric transmission needs the exposure's PWV and airmasses, its amplitude
# priors need the pointing and timestamp, and its science-line mask is centred
# on the Halpha velocity measured from the science spectrum of the SAME
# exposure.  So a raw sky spectrum is decomposed by putting it, with its
# exposure's header, into an in-memory one-row stack (`build_one_row_stack`, §3) and
# running `decompose_parallel.decompose_in_process` -- the same code and the
# same defaults as the cluster run that built the training corpus.
#
# Bind these before running (the defaults decompose the SkyE median from §3):
#   raw_flux   : (n_pix,) sky flux in erg/s/cm^2/A on `wave`
#   raw_lsf    : (n_pix,) its LSF FWHM in A
#   raw_header : the exposure's primary header (pointings, OBSTIME, airmasses, PWV)
# The spectrum goes into the SkyE slot; the other two slots keep this
# exposure's own spectra so the mask and the near/far assignment are unchanged.
raw_flux = globals().get("raw_flux", flux_east)
raw_lsf = globals().get("raw_lsf", skye_lsf_fwhm_ang)
raw_header = globals().get("raw_header", hdr)

_raw_stack, _near_raw, _ = build_one_row_stack(
    raw_header,
    spectra={"Sci": flux_sci_med, "SkyE": raw_flux, "SkyW": flux_west},
    lsfs={"Sci": sci_lsf_fwhm_ang, "SkyE": raw_lsf, "SkyW": skyw_lsf_fwhm_ang},
    fiber_counts={"Sci": sci_fibers.sum(), "SkyE": skye_fibers.sum(), "SkyW": skyw_fibers.sum()})
_kind = "sky1" if _near_raw == "SkyE" else "sky2"
fit, _flags = _dp.decompose_in_process(_raw_stack, rows=(0,), kinds=(_kind,))[(_kind, 0)]
print(f"fit_status:   {fit.fit_status}")
print(f"chi2_red:     {fit.reduced_chi2:.3f}   (photon-weighted, as in the corpus)")
print(f"R^2:          {fit.r2:.4f}")
print(f"elapsed:      {fit.fit_elapsed_sec:.1f}s")
print(f"reliability:  {_flags.get('reliability')}")
print(f"components:   {list(fit.components)}")
print(f"coef.shape:   {fit.coef.shape}  (matches predict_sky_from_minimal_inputs input dim)")

import plotly.graph_objects as go

# Plot the observed flux, the total best-fit, and the per-component
# contributions on one axis (all in the decomposition's units, flux x FACTOR).
_palette = {
    'oh':      '#e6194b',
    'moon':    '#f58231',
    'zodi':    '#ffe119',
    'diffuse': '#3cb44b',
    'atom':    '#9a6324',
    'orc':     '#911eb4',
    'o2':      '#42d4f4',
}
fig = go.Figure()
fig.add_trace(go.Scatter(x=wave, y=np.asarray(raw_flux) * FACTOR, mode='lines',
                          line=dict(color='rgba(120,120,120,0.55)', width=0.8),
                          name='observed'))
fig.add_trace(go.Scatter(x=wave, y=fit.bestfit_lsf, mode='lines',
                          line=dict(color='black', width=1.1),
                          name='best-fit total'))
for _name in ('moon', 'zodi', 'diffuse', 'oh', 'atom', 'orc', 'o2'):
    _comp = fit.components.get(_name)
    if _comp is None:
        continue
    fig.add_trace(go.Scatter(x=wave, y=np.asarray(_comp), mode='lines',
                              line=dict(color=_palette[_name], width=0.6),
                              name=_name))
fig.update_layout(xaxis_title='wavelength [Å]',
                   yaxis_title='flux (native decomp units)',
                   height=520, margin=dict(l=60, r=20, t=40, b=50),
                   title=f'Decomposed sky spectrum '
                         f'(chi2_red={fit.reduced_chi2:.2f}, R2={fit.r2:.4f})')
fig.show()